# Q3 — C4.5 Rule Generation (Play Dataset)

Generate rules from a trained decision tree (C4.5 style), compute coverage and accuracy for each rule, and compute overall accuracy.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, _tree
import numpy as np

# Play Dataset (classic Play Tennis example)
data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
    'Windy': ['False','True','False','False','False','True','True','False','False','False','True','True','False','True'],
    'Play': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
}
df = pd.DataFrame(data)
df.head()

,Outlook,Temperature,Humidity,Windy,Play
0,Sunny,Hot,High,False,No
1,Sunny,Hot,High,True,No
2,Overcast,Hot,High,False,Yes
3,Rain,Mild,High,False,Yes
4,Rain,Cool,Normal,False,Yes


In [4]:
# Train decision tree using entropy (C4.5 uses information gain ratio; here we use entropy-based tree for rule extraction)
X = pd.get_dummies(df.drop(columns=['Play']))
y = df['Play']
clf = DecisionTreeClassifier(criterion='entropy', random_state=0)
clf.fit(X, y)

# Function to extract rules from sklearn tree
def tree_to_rules(clf, feature_names):
    tree = clf.tree_
    paths = []
    path = []
    def recurse(node, path):
        if tree.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_names[tree.feature[node]]
            threshold = tree.threshold[node]
            # For one-hot features, threshold will be 0.5 typically; extract the column name and value
            path_left = path.copy()
            path_left.append((name, "<=", threshold))
            recurse(tree.children_left[node], path_left)
            path_right = path.copy()
            path_right.append((name, ">", threshold))
            recurse(tree.children_right[node], path_right)
        else:
            # leaf
            value = tree.value[node]
            classes = clf.classes_
            class_idx = np.argmax(value)
            paths.append((path, classes[class_idx], value.flatten().tolist()))
    recurse(0, [])
    return paths

rules = tree_to_rules(clf, X.columns.tolist())

# Convert rules into readable form and compute coverage & accuracy
def rule_matches(rule, X_row):
    for feat, op, thr in rule:
        val = X_row.get(feat, 0)
        if op == "<=":
            if not (val <= thr + 1e-8):
                return False
        else:
            if not (val > thr - 1e-8):
                return False
    return True

readable_rules = []
for rule, pred_class, leaf_counts in rules:
    # compute coverage and accuracy
    matches = []
    for i, row in X.iterrows():
        if rule_matches(rule, row.to_dict()):
            matches.append(i)
    coverage = len(matches)
    if coverage > 0:
        acc = (y.iloc[matches] == pred_class).mean()
    else:
        acc = None
    readable_rules.append({'rule': rule, 'pred': pred_class, 'coverage': coverage, 'accuracy': acc})

Finally compute the overall model performance (actual vs predicted)

In [3]:
# Overall model performance
y_pred = clf.predict(X)
df_res = df.copy()
df_res['Predicted'] = y_pred
from sklearn.metrics import accuracy_score
acc_overall = accuracy_score(df_res['Play'], df_res['Predicted'])
print('Overall accuracy:', acc_overall)
display(df_res)

Overall accuracy: 1.0


,Outlook,Temperature,Humidity,Windy,Play,Predicted
0,Sunny,Hot,High,False,No,No
1,Sunny,Hot,High,True,No,No
2,Overcast,Hot,High,False,Yes,Yes
3,Rain,Mild,High,False,Yes,Yes
4,Rain,Cool,Normal,False,Yes,Yes
5,Rain,Cool,Normal,True,No,No
6,Overcast,Cool,Normal,True,Yes,Yes
7,Sunny,Mild,High,False,No,No
8,Sunny,Cool,Normal,False,Yes,Yes
9,Rain,Mild,Normal,False,Yes,Yes
